# CORDEX subset memory failure

This notebook reproduces a CDS WPS workflow that used too much memory while subsetting daily CORDEX `tas` data for 2056–2075.

The request selects every month and every year in the time range, so it effectively keeps the complete 20-year daily time series over the full spatial domain. The result is deliberately **not** opened with `resp.datasets()` in this notebook, because doing so could add client-side memory use to the server-side issue under investigation.


## Original WPS workflow

The payload below is copied from the failing request.


In [1]:
request = {
    "inputs": {
        "tas": [
            "c3s-cordex.output.EUR-11.CLMcom.MPI-M-MPI-ESM-LR.rcp85."
            "r1i1p1.CLMcom-CCLM4-8-17.v1.day.tas.v20140515"
        ]
    },
    "steps": {
        "subset_tas_1": {
            "run": "subset",
            "in": {
                "collection": "inputs/tas",
                "time_components": (
                    "month:jan,feb,mar,apr,may,jun,jul,aug,sep,oct,nov,dec|"
                    "year:2056,2057,2058,2059,2060,2061,2062,2063,2064,"
                    "2065,2066,2067,2068,2069,2070,2071,2072,2073,2074,2075"
                ),
                "time": "2056/2075",
            },
        }
    },
    "outputs": {"output": "subset_tas_1/output"},
    "doc": "workflow",
}

request


{'inputs': {'tas': ['c3s-cordex.output.EUR-11.CLMcom.MPI-M-MPI-ESM-LR.rcp85.r1i1p1.CLMcom-CCLM4-8-17.v1.day.tas.v20140515']},
 'steps': {'subset_tas_1': {'run': 'subset',
   'in': {'collection': 'inputs/tas',
    'time_components': 'month:jan,feb,mar,apr,may,jun,jul,aug,sep,oct,nov,dec|year:2056,2057,2058,2059,2060,2061,2062,2063,2064,2065,2066,2067,2068,2069,2070,2071,2072,2073,2074,2075',
    'time': '2056/2075'}}},
 'outputs': {'output': 'subset_tas_1/output'},
 'doc': 'workflow'}

## Build the equivalent Rooki workflow

Importing Rooki contacts the configured WPS service. Change `ROOK_URL` here if the reproduction should run against another deployment.


In [2]:
import json
import os

os.environ["ROOK_URL"] = "http://rook.dkrz.de/wps"

from rooki import operators as ops


In [3]:
tas = ops.Input("tas", request["inputs"]["tas"])
subset = ops.Subset(
    tas,
    time="2056/2075",
    #time_components=(
    #                "month:jan,feb,mar,apr,may,jun,jul,aug,sep,oct,nov,dec|"
    #                "year:2056,2057,2058,2059,2060,2061,2062,2063,2064,2065,2066,2067,2068,2069,2070"
    #),
)

serialized_request = json.loads(subset._serialise())
serialized_request


{'inputs': {'tas': ['c3s-cordex.output.EUR-11.CLMcom.MPI-M-MPI-ESM-LR.rcp85.r1i1p1.CLMcom-CCLM4-8-17.v1.day.tas.v20140515']},
 'steps': {'subset_tas_1': {'run': 'subset',
   'in': {'collection': 'inputs/tas', 'time': '2056/2075'}}},
 'outputs': {'output': 'subset_tas_1/output'},
 'doc': 'workflow'}

## Reproduce the failure

The next cell submits the full request and may consume substantial memory on the Rook server. Run it only against the deployment being tested.


In [4]:
resp = subset.orchestrate()
resp.ok, resp.status


(True, 'ProcessSucceeded')

## Observed result on the test Rook server

On 2026-08-11, the request exceeded its Slurm cgroup memory allocation on a test Rook server. The host has 32 GB of physical memory, but Slurm advertises 25,466 MB (about 24.9 GiB) as `RealMemory`. The `fast` partition defaults to 4,244 MB (about 4.1 GiB) per job and permits at most 25,466 MB per job. Unless the submitted job explicitly requested more memory, its cgroup limit was therefore 4,244 MB rather than the host's full memory.

Relevant Slurm configuration:

```ini
SelectType=select/cons_res
SelectTypeParameters=CR_Core_Memory
JobAcctGatherType=jobacct_gather/cgroup
ProctrackType=proctrack/cgroup
TaskPlugin=task/affinity,task/cgroup
NodeName=localhost CPUs=6 RealMemory=25466
PartitionName=fast Default=YES DefaultTime=90 DefMemPerNode=4244 MaxMemPerNode=25466 MaxTime=90 Nodes=localhost
```

Slurm accounting (`sacct`) is not available on this test node. A rerun, job 26629, remained visible to `scontrol show job -dd 26629` and confirmed that the default allocation applies: one CPU and 4,244 MB of memory. It entered `OUT_OF_MEMORY` after 53 seconds. The relevant fields were:

```text
JobId=26629 JobState=OUT_OF_MEMORY Reason=OutOfMemory ExitCode=0:125
RunTime=00:00:53
NumNodes=1 NumCPUs=1 NumTasks=1 CPUs/Task=1
TRES=cpu=1,mem=4244M,node=1,billing=1
Nodes=localhost CPU_IDs=0 Mem=4244 GRES=
MinCPUsNode=1 MinMemoryNode=4244M MinTmpDiskNode=0
```

This confirms a reproducible OOM at the job's 4,244 MB cgroup limit; it does not show that the operation requires all 25–32 GB available on the node. Slurm reported a cgroup out-of-memory kill. The preceding xarray message is a `FutureWarning`; the terminating failure is the OOM kill.

```text
/usr/local/anaconda/envs/rook/lib/python3.13/site-packages/clisops/utils/dataset_utils.py:448: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(dset, **multi_file_kwargs)
/var/spool/slurm/d/job26621/slurm_script: line 2: 255245 Killed                  /usr/local/Miniforge3-24.9.2-0-Linux-x86_64/envs/rook/bin/joblauncher '-c' '/var/lib/pywps/tmp/rook/pywps_process_scs7bo4y/pywps.cfg' '/var/lib/pywps/tmp/rook/pywps_process_scs7bo4y/job_6akzgrzr.dump'
slurmstepd: error: Detected 1 oom-kill event(s) in StepId=26621.batch. Some of your processes may have been killed by the cgroup out-of-memory handler.
```

### Comparison tests

All tests used the same dataset and Slurm configuration shown above. Job 26629 confirms that jobs without an explicit memory request receive the configured 4,244 MB default.

| `time` | `time_components` | Result |
|---|---|---|
| `2056/2075` | omitted | Succeeds and returns quickly |
| `2056/2075` | all months and all years in the range | Exceeds the 4,244 MB cgroup allocation; OOM kill |
| `2040/2080` | omitted | Exceeds the 4,244 MB cgroup allocation; OOM kill |

For 2056–2075, `time_components` is logically redundant because it names every month and every year already selected by `time`. Nevertheless, including it raises peak memory enough to cross the job's cgroup limit. Without `time_components`, increasing the time range to 2040–2080 also crosses that limit.

This suggests two related effects to investigate: baseline memory use grows with the selected time span, and the `time_components` selection path creates substantial additional memory pressure even when it does not reduce the result. The warning location shows that `open_mfdataset` was called, but does not by itself prove which later operation held or allocated the memory at the point of the OOM kill.


## Inspect the response without loading data

If the workflow succeeds, list the output URLs without downloading or opening the NetCDF result. If it fails, displaying `resp` preserves the response details for diagnosis.


In [5]:
resp


Metalink URL: http://rook7.cloud.dkrz.de:80/outputs/rook/63c0e83e-958b-11f1-be32-fa163eb671ca/input.meta4, num files: 4

In [6]:
if resp.ok:
    print("Output URLs (not downloaded):")
    for url in resp.download_urls():
        print(url)


Output URLs (not downloaded):
https://data.mips.climate.copernicus.eu/thredds/fileServer/esg_c3s-cordex/output/EUR-11/CLMcom/MPI-M-MPI-ESM-LR/rcp85/r1i1p1/CLMcom-CCLM4-8-17/v1/day/tas/v20140515/tas_EUR-11_MPI-M-MPI-ESM-LR_rcp85_r1i1p1_CLMcom-CCLM4-8-17_v1_day_20560101-20601231.nc
https://data.mips.climate.copernicus.eu/thredds/fileServer/esg_c3s-cordex/output/EUR-11/CLMcom/MPI-M-MPI-ESM-LR/rcp85/r1i1p1/CLMcom-CCLM4-8-17/v1/day/tas/v20140515/tas_EUR-11_MPI-M-MPI-ESM-LR_rcp85_r1i1p1_CLMcom-CCLM4-8-17_v1_day_20610101-20651231.nc
https://data.mips.climate.copernicus.eu/thredds/fileServer/esg_c3s-cordex/output/EUR-11/CLMcom/MPI-M-MPI-ESM-LR/rcp85/r1i1p1/CLMcom-CCLM4-8-17/v1/day/tas/v20140515/tas_EUR-11_MPI-M-MPI-ESM-LR_rcp85_r1i1p1_CLMcom-CCLM4-8-17_v1_day_20660101-20701231.nc
https://data.mips.climate.copernicus.eu/thredds/fileServer/esg_c3s-cordex/output/EUR-11/CLMcom/MPI-M-MPI-ESM-LR/rcp85/r1i1p1/CLMcom-CCLM4-8-17/v1/day/tas/v20140515/tas_EUR-11_MPI-M-MPI-ESM-LR_rcp85_r1i1p1_CLMcom-CCLM4